# Week 8 — Data + MLP Inverse Dynamics (Colab)

Trains the black-box MLP inverse-dynamics baseline for the *Learned Robot Arm Dynamics + Control* project.

**MuJoCo runs headless here**, so data generation and training both happen in this notebook — no local simulator needed. The MLP is small, so a **CPU runtime is fine** (a GPU mainly helps in Week 9's DeLaN).

Flow: clone repo → install deps → headless MuJoCo → generate data → train MLP → evaluate (in-dist vs OOD) → download `mlp.pt`.

> **Set `REPO_URL` in the first code cell before running.**

## 1. Clone the project repo
Re-running this cell later does a `git pull`, so pushing a change locally and re-running picks it up — no re-upload.

In [ ]:
import os

# EDIT THIS: your GitHub repo (public = no auth needed).
REPO_URL = "https://github.com/<your-username>/robot_arm_delan.git"
REPO_DIR = "robot_arm_delan"

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

# Make the project importable and switch into it.
import sys
PROJECT = os.path.abspath(REPO_DIR)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
os.chdir(PROJECT)
print("working dir:", os.getcwd())
print("contents:", sorted(os.listdir())[:20])

## 2. Install dependencies
Colab already has `torch` + `numpy` + `matplotlib`; we add `mujoco`. (`imageio` is preinstalled.)

In [ ]:
!pip install -q mujoco>=3.0

import torch, numpy as np, mujoco
print("torch  :", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("numpy  :", np.__version__)
print("mujoco :", mujoco.__version__)

## 3. Headless MuJoCo
Data generation needs no display, but set the EGL backend so any rendering (e.g. a figure-eight video) also works on Colab's headless GPU. Must be set **before** MuJoCo creates a rendering context.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"   # headless OpenGL; harmless if we never render

# Sanity check: the model loads and steps under MuJoCo.
from arm_sim import ArmSim
sim = ArmSim(backend="mujoco")
sim.reset([0.5, 0.5], [0.0, 0.0])
sim.set_torque([1.0, -0.5]); sim.step()
print("MuJoCo backend OK — q after one step:", sim.q)

## 4. Generate datasets
Writes `data/{train,test_id,test_ood}.npz`. `test_ood` is deliberately faster/stronger than training — the regime where the black-box MLP should struggle.

In [ ]:
from data_collection import make_all
make_all(backend="mujoco", seed=0)

## 5. Train the MLP
Auto-uses GPU if the runtime has one (not required). Saves `models/mlp.pt` and prints torque-prediction RMSE for in-distribution vs OOD.

In [ ]:
from train_mlp import train
device = "cuda" if torch.cuda.is_available() else "cpu"
print("training on:", device)
model = train(epochs=300, lr=1e-3, batch=256, device=device)

## 6. Closed-loop evaluation — MLP vs analytic
The headline Week-8 result. Analytic should be tiny in both regimes; a good MLP is close under `nominal` and degrades under `fast` (OOD). We use the analytic backend here (fast and deterministic); pass `--backend mujoco` behavior by setting `backend='mujoco'` if you prefer the full sim.

In [ ]:
import dynamics
from mlp_model import MLPInverseDynamics
from evaluate_model import closed_loop_track, REGIMES

mlp = MLPInverseDynamics.load("models/mlp.pt")
models = {"analytic": dynamics, "mlp": mlp}

BACKEND = "analytic"  # switch to "mujoco" for the full simulator
print(f"End-effector RMS tracking error (mm), backend={BACKEND}\n")
print(f"{'model':10s}" + "".join(f"{r:>12s}" for r in REGIMES))
print("-" * 34)
for name, m in models.items():
    row = f"{name:10s}"
    for rname, regime in REGIMES.items():
        rms = closed_loop_track(m, BACKEND, regime)
        row += f"{rms*1000:12.3f}"
    print(row)

## 7. Plot the extrapolation gap
Torque-prediction error per sample, in-distribution vs OOD — the visual that motivates DeLaN.

In [ ]:
import matplotlib.pyplot as plt
from train_mlp import load_split

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, split in zip(axes, ["test_id", "test_ood"]):
    X, Y = load_split(split)
    pred = mlp.predict(X)
    err = np.linalg.norm(pred - Y, axis=1)
    ax.hist(err, bins=60)
    ax.set_title(f"{split}   RMSE={np.sqrt((err**2).mean()):.3f} N·m")
    ax.set_xlabel("per-sample torque error (N·m)")
axes[0].set_ylabel("count")
fig.suptitle("MLP inverse-dynamics error: in-distribution vs OOD")
fig.tight_layout(); plt.show()

## 8. Download the trained model
Drop `mlp.pt` into your local `robot_arm_delan/models/` to run the benchmark offline. (Or mount Drive and save there.)

In [ ]:
from google.colab import files
files.download("models/mlp.pt")